# TESTING NEW DATA_PREP_FILE


In [10]:
from utils.centralized_training import aggregate_datasets
from utils.data import save_dataset

from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.preprocessing import OneHotEncoder, MinMaxScaler
import matplotlib.pyplot as plt
from typing import Tuple
import seaborn as sns
import pandas as pd
import numpy as np
import joblib
import json
import os

In [11]:
# Function to import data
def import_data(path):
  df = pd.DataFrame()
  for node_type in node_types:
      # Retrieve all files in the data folder
      file_csv = [file for file in os.listdir(path + node_type) if file.endswith('.csv')]
      # Create the dataframe by concatenating all read files
      dataframes = []
      for file in file_csv:
          file_path = os.path.join(path + node_type, file)
          df_temp = pd.read_csv(file_path)
          # Remove the columns in the dataframe that begin with "function_"
          df_temp.drop(columns=[col for col in df_temp if col.startswith('function_')], inplace=True)
          # Add the column "node_type" and assign the value of 'type' to all rows
          if node_type == "HEAVY":
              df_temp["node_type"] = 0
          elif node_type == "MID":
              df_temp["node_type"] = 1
          else:
              df_temp["node_type"] = 2

          dataframes.append(df_temp)

      df = pd.concat([df, *dataframes], axis=0, ignore_index=True)

  return df


# Function used to fill NaN values within the dataframe X
def fill_NaN(X):
  for col in X:
    if(col.startswith('success_rate_')):
      X.loc[:, col] = X.loc[:, col].fillna(1)
    else:
      X.loc[:, col] = X.loc[:, col].fillna(0)
  return X

# Function to assign zero to noise metrics where requests rates are 0
def assign_zero_to_noise(df):
  functions = [col[14:] for col in df if col.startswith('rate')]

  for function in functions:
      df.loc[df['rate_function_' + function] == 0, ['cpu_usage_function_' + function, 'ram_usage_function_' + function, 'power_usage_function_' + function, 'replica_' + function]] = 0
  return df

# Function to select relevant columns
def select_columns(df):
  targets = [col for col in df if (col.startswith('cpu_usage_') or
                                   col.startswith('ram_usage_') or
                                   col.startswith('overloaded_node')
                                   or col.startswith('medium_latency')) and 'idle' not in col]# or col.startswith('replica')
  features = [col for col in df if col.startswith('rate_') or 'node_type' in col]
  features.sort()
  df = df[features+targets]
  return df

# Function to Rename Eat Memory column's Name
def rename_eat_memory(df):
  new_column_name = {i:i.replace("-","_") for i in df.keys() if "eat" in i}
  df.rename(columns=new_column_name, inplace=True)
  return df

# Function to add overload status and overload ratio.

# 0 = all 0, 1 = all 1, 2 = mix
def group_status(x):
    if (x == 0).all():
        return 0
    elif (x == 1).all():
        return 1
    else:
        return 2
    
def overload_status_ratio(df,features,target):
  grouped = df.groupby(features)

  df['overloaded_status'] = grouped[target].transform(group_status)

  # Ratio of 1’s in the group
  df['overloaded_ratio'] = grouped[target].transform('mean')

  return df

def get_node_capacity(node_id: int) -> int:
  # Lookup for node capacities
  GB = 1024 ** 3  # 1 GB in bytes
  capacities = {
      0: 24 * GB,   # Heavy
      1: 16 * GB,   # Mid
      2:  8 * GB    # Light
  }
  return capacities[node_id]

def ram_usage_to_percentage(ram_usage, node_type):
    """Return RAM utilization % given usage (bytes) and node type."""
    return (ram_usage / get_node_capacity(node_type)) * 100

# Function to compute theoretical RAM usage percentage.
def compute_ram_usage_percentage_theoretical(df, ram_col="ram_usage_node", node_col="node_type"):
    """Apply ram_usage_to_perc on the DataFrame."""
    # df["ram_usage_node_perc_theor"] = df.apply(
    #     lambda row: ram_usage_to_percentage(row[ram_col], row[node_col]),
    #     axis=1
    # )
    # Compute values
    perc_values = df.apply(
        lambda row: ram_usage_to_percentage(row[ram_col], row[node_col]),
        axis=1
    )

    # Find the index of ram_usage column
    insert_at = df.columns.get_loc("ram_usage_node_percentage") + 1

    # Insert new column at that position
    df.insert(insert_at, "ram_usage_node_perc_theor", perc_values)

    return df


# ci sono queste costanti da trasformare in funzioni??-----------------

# 🎯 Define target columns
#target_prefixes = ("cpu_usage_", "ram_usage_", "overloaded_node", "medium_latency")
#targets = [
   # col for col in df_source.columns
    #if col.startswith(target_prefixes) and "idle" not in col
#]

# 🔢 Numerical & Categorical Features
#categorical_features = ["node_type"]
#numerical_features = [col for col in df_source.columns if col.startswith("rate_")]
#features = categorical_features + numerical_features

# 🧭 Target Categorization
#categorical_targets = ["overloaded_node"]
#numerical_targets = [col for col in targets if col not in categorical_targets]

#Remove rows where all specified columns have value 0.
def remove_baselines(df,features):
    df = df.copy()
    return df[~(df[features] == 0).all(axis=1)]

# Function to remove outliers
def remove_outliers(df):
  # Iterate over each target column and handle outliers
  functions_column = [col for col in df if col.startswith('rate')]
  targets = [col for col in df if (col.startswith('power_usage_') or col.startswith('cpu_usage_') or col.startswith('ram_usage_') or col.startswith('overloaded_node') or col.startswith('medium_latency')) and 'idle' not in col]
  grouped = df.groupby(functions_column + ['node_type'])
  threshold = 1
  for target in targets:
      print(target)
      if target != 'overloaded_node':
          mean = grouped[target].transform('mean')
          std = grouped[target].transform('std')
          outliers = (df[target] > mean + threshold * std) | (df[target] < mean - threshold * std)
          print(outliers.sum())
          df[target] = df[target].where(~outliers, mean)
      else:
          # Replace the overloaded value of the group by the mode.
          new_overloaded = grouped[target].transform(lambda x: x.mode().iloc[0])
          df['overloaded_node'] = new_overloaded
          print(df["overloaded_node"].value_counts())
  df_only_useful = df[functions_column + targets]
  return df

In [12]:
# data preparation
def prepare_single_dataset(path_to_csvs: str, features: list) -> pd.DataFrame:
    df = import_data(path_to_csvs)
    df = fill_NaN(df)
    df = assign_zero_to_noise(df)
    df = select_columns(df)
    df = rename_eat_memory(df)
    df = overload_status_ratio(df, features, "overloaded_node")
    df = compute_ram_usage_percentage_theoretical(
        df,
        ram_col="ram_usage_node",
        node_col="node_type"
    )
    return df

def clean_for_model(df: pd.DataFrame, numerical_features: list) -> pd.DataFrame:
    df = remove_baselines(df, numerical_features)
    df = remove_outliers(df)
    return df
# build task and dataset dictionaries
def build_tasks_and_datasets(df: pd.DataFrame):
    target_prefixes = ("cpu_usage_", "ram_usage_", "overloaded_node", "medium_latency")
    targets = [
        col for col in df.columns
        if col.startswith(target_prefixes) and "idle" not in col
    ]

    categorical_features = ["node_type"]
    numerical_features = [col for col in df.columns if col.startswith("rate_")]
    features = categorical_features + numerical_features

    categorical_targets = ["overloaded_node"]
    numerical_targets = [col for col in targets if col not in categorical_targets]

    tasks = {
        "Multi_Task_regression": {
            "features": features,
            "targets": numerical_targets
        },
        "Multi_Task_classification": {
            "features": features,
            "targets": categorical_targets
        }
    }

    tasks_unified = {
        "Multi_Task": {
            "features": features,
            "targets": numerical_targets + categorical_targets,
            "regression_targets": numerical_targets,
            "classification_targets": categorical_targets
        }
    }

    return features, numerical_features, tasks, tasks_unified

# final data preparation function
def prepare_source_target_datasets(path_source: str, path_target: str):
    initial_features = [
        "node_type",
        "rate_function_curl",
        "rate_function_eat_memory",
        "rate_function_env",
        "rate_function_figlet",
        "rate_function_nmap",
        "rate_function_shasum",
    ]

    df_source = prepare_single_dataset(path_source, initial_features)
    df_target = prepare_single_dataset(path_target, initial_features)

    features, numerical_features, tasks, tasks_unified = build_tasks_and_datasets(df_source)

    df_source = clean_for_model(df_source, numerical_features)
    df_target = clean_for_model(df_target, numerical_features)

    feature_dataset_source, target_dataset_source = prepare_feature_target_datasets_cv(
        df_source, tasks_unified
    )
    feature_dataset_target, target_dataset_target = prepare_feature_target_datasets_cv(
        df_target, tasks_unified
    )

    return {
        "df_source": df_source,
        "df_target": df_target,
        "features": features,
        "numerical_features": numerical_features,
        "tasks": tasks,
        "tasks_unified": tasks_unified,
        "feature_dataset_source": feature_dataset_source,
        "target_dataset_source": target_dataset_source,
        "feature_dataset_target": feature_dataset_target,
        "target_dataset_target": target_dataset_target,
    }

def prepare_feature_target_datasets_cv(df, tasks):
    """
    Prepare datasets for multiple targets regression and overloaded node classification
    with optional SMOTENC oversampling.

    Parameters
    ----------
    df : pandas.DataFrame
        Full dataset containing both features and targets.
    features : list of str
        List of feature column names.
    targets : list of str
        List of target column names.
    categorical_features : list of str
        Categorical feature columns to be passed to SMOTENC.

    Returns
    -------
    features_datasets : dict
        Mapping target names to feature DataFrames (oversampled if needed).
    target_datasets : dict
        Mapping target names to target Series.
    """

    feature_dataset = {}
    target_dataset = {}
    df_augmented = None
    for task_name, task_info in tasks.items():
      X = df[task_info["features"]]
      y = df[task_info["targets"]]
      feature_dataset[task_name] = X
      target_dataset[task_name] = y

    return feature_dataset, target_dataset


In [6]:
path_to_csvs = "/Users/kingsley/Documents/TESI/GITHUB_TESI/gl-forecasting-edge-performance/data/raw/source_domain/"
path_to_csvs_target = "/Users/kingsley/Documents/TESI/GITHUB_TESI/gl-forecasting-edge-performance/data/raw/target_domain/"
node_types = ["LIGHT", "MID", "HEAVY"]

In [7]:
result = prepare_source_target_datasets(path_to_csvs, path_to_csvs_target)

/var/folders/0r/2hzs6flj5f39y43jzjh230300000gn/T/ipykernel_19751/531831007.py:24: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat([df, *dataframes], axis=0, ignore_index=True)
/var/folders/0r/2hzs6flj5f39y43jzjh230300000gn/T/ipykernel_19751/531831007.py:35: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  X.loc[:, col] = X.loc[:, col].fillna(0)


cpu_usage_function_shasum
23887
ram_usage_function_shasum
23430
medium_latency_function_shasum
23610
cpu_usage_function_curl
14017
ram_usage_function_curl
13650
medium_latency_function_curl
13920
cpu_usage_function_eat_memory
4571
ram_usage_function_eat_memory
4501
medium_latency_function_eat_memory
4564
cpu_usage_node
32741
ram_usage_node
32385
ram_usage_node_percentage
32127
ram_usage_node_perc_theor
32385
overloaded_node
overloaded_node
0.0    89169
1.0     9563
Name: count, dtype: int64
cpu_usage_function_nmap
7319
ram_usage_function_nmap
7245
medium_latency_function_nmap
7301
cpu_usage_function_env
19397
ram_usage_function_env
18812
medium_latency_function_env
19202
cpu_usage_function_figlet
19900
ram_usage_function_figlet
19762
medium_latency_function_figlet
19790
cpu_usage_function_figlet
16476
ram_usage_function_figlet
16403
medium_latency_function_figlet
16250
cpu_usage_function_shasum
16498
ram_usage_function_shasum
16352
medium_latency_function_shasum
16298
cpu_usage_functio

In [13]:
df_source = result["df_source"]

In [14]:
df_source

,node_type,rate_function_curl,rate_function_eat_memory,rate_function_env,rate_function_figlet,rate_function_nmap,rate_function_shasum,cpu_usage_function_shasum,ram_usage_function_shasum,medium_latency_function_shasum,...,ram_usage_function_nmap,medium_latency_function_nmap,cpu_usage_function_env,ram_usage_function_env,medium_latency_function_env,cpu_usage_function_figlet,ram_usage_function_figlet,medium_latency_function_figlet,overloaded_status,overloaded_ratio
0,2,60.0,0.0,0.0,0.0,0.0,100.0,7.691,4042069.333,37454241.5,...,0.000000e+00,0.000000e+00,0.000000,0.000000e+00,0.000000e+00,0.000000,0.000,0.000000e+00,0,0.0
1,2,60.0,0.0,0.0,0.0,0.0,100.0,7.751,9199274.667,16880204.0,...,0.000000e+00,0.000000e+00,0.000000,0.000000e+00,0.000000e+00,0.000000,0.000,0.000000e+00,0,0.0
2,2,60.0,0.0,0.0,0.0,0.0,100.0,8.378,9214634.667,35575151.0,...,0.000000e+00,0.000000e+00,0.000000,0.000000e+00,0.000000e+00,0.000000,0.000,0.000000e+00,0,0.0
3,2,60.0,10.0,0.0,0.0,0.0,100.0,8.261,1366357.333,76360433.0,...,0.000000e+00,0.000000e+00,0.000000,0.000000e+00,0.000000e+00,0.000000,0.000,0.000000e+00,1,1.0
4,2,60.0,10.0,0.0,0.0,0.0,100.0,7.820,1323008.000,87032021.0,...,0.000000e+00,0.000000e+00,0.000000,0.000000e+00,0.000000e+00,0.000000,0.000,0.000000e+00,1,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
98779,0,0.0,0.0,20.0,70.0,70.0,0.0,0.000,0.000,0.0,...,2.022266e+08,1.655140e+10,2.024333,1.616555e+06,1.278133e+07,10.196000,5607879.111,1.611891e+07,0,0.0
98780,0,0.0,0.0,20.0,70.0,70.0,0.0,0.000,0.000,0.0,...,3.036606e+08,1.697898e+10,2.011000,1.553067e+06,1.207876e+07,10.012000,1713152.000,1.596136e+07,0,0.0
98781,0,0.0,0.0,30.0,70.0,70.0,0.0,0.000,0.000,0.0,...,2.909239e+08,1.728170e+10,2.889333,1.520640e+06,1.444850e+07,9.990167,7250379.020,1.522533e+07,0,0.0
98782,0,0.0,0.0,30.0,70.0,70.0,0.0,0.000,0.000,0.0,...,2.953576e+08,1.793603e+10,2.822000,1.727829e+06,1.444850e+07,9.619000,7250379.020,2.362620e+07,0,0.0


In [ ]:
# df_source = result["df_source"]

# output_path = "/Users/kingsley/Documents/TESI/TEST_RESULTS/TESTS/df_source.csv"

# df_source.to_csv(output_path, index=False)
# print(f"Salvato in: {output_path}")

# TEST THE NODE DISTRIBUTION

In [15]:

def convert_node_type(node_type: str) -> int:
  node_type_idx = -1
  if node_type == "HEAVY":
    node_type_idx = 0
  elif node_type == "MID":
    node_type_idx = 1
  elif node_type == "LIGHT":
    node_type_idx = 2
  else:
    raise RuntimeError(f"Node type `{node_type}` does not exist")
  return node_type_idx

def assign_node_type(
    towers: pd.DataFrame, node_types: list, rng: np.random.Generator
  ) -> pd.DataFrame:
  nt = [
    convert_node_type(rng.choice(node_types)) for _ in range(len(towers))
  ]
  towers["node_type"] = nt
  return towers

def assign_functions( # attualmente sembra non essere utilizzato 
    towers: pd.DataFrame, 
    functions: list, 
    n_functions_per_node: int,
    rng: np.random.Generator
  ) -> pd.DataFrame:
  functions_per_node = [
    [
      rng.choice(functions) for _ in range(n_functions_per_node)
    ] for _ in range(len(towers))
  ]
  towers["functions"] = functions_per_node
  return towers

In [60]:
def load_network(
    path_to_networks: str, 
    n: int, 
    k: int, 
    #seed: int, 
    simulation: int, 
    rng: np.random.Generator, 
    node_types: list = None,
    functions: list = None,
    n_functions_per_node: int = None
  ):
  towers_file = os.path.join(
    path_to_networks, f"porto_{n}n_{k}k/{simulation}/towers.csv"#path_to_networks, f"porto_{n}n_{k}k/seed{seed}/{simulation}/towers.csv"
  )
  towers = pd.read_csv(towers_file)
  if node_types is not None:
    towers = assign_node_type(towers, node_types, rng)
  if functions is not None:
    towers = assign_functions(towers, functions, n_functions_per_node, rng)
  return towers

In [17]:
# funzione per distribuire i dati ai nodi

def build_nodes_dataframe(
    data_x: pd.DataFrame, data_y: pd.DataFrame, towers: pd.DataFrame
  ) -> dict:
  nodes_dataset = {}
  surely_test_set = {}
  for node_type, node_type_data in data_x.groupby("node_type"):
    # extract the corresponding y values
    node_type_targets = data_y.loc[node_type_data.index]
    # count nodes that have the required type
    nnodes = len(towers[towers["node_type"] == node_type])
    # count available data for that node type (dividing overloaded and 
    # not overloaded)
    nvals = len(node_type_data)
    nvals_overloaded = 0
    if "overloaded_node" in node_type_targets:
      nvals_overloaded = len(
        node_type_targets[node_type_targets["overloaded_node"] == 1]
      )
    nvpn, remainder = [None, None], [None, None]
    nvpn[0], remainder[0] = divmod(nvals - nvals_overloaded, nnodes)
    nvpn[1], remainder[1] = divmod(nvals_overloaded, nnodes)
    # split equally
    for i, node_id in enumerate(
        towers[towers["node_type"] == node_type].index
      ):
      for overload_status in [0, 1]:
        idxs = node_type_targets[
          node_type_targets["overloaded_node"] == overload_status
        ].iloc[
          i * nvpn[overload_status] : (i+1) * nvpn[overload_status]
        ].index
        # add
        if node_id not in nodes_dataset:
          nodes_dataset[node_id] = {
            "x": node_type_data.loc[idxs,:],
            "y": node_type_targets.loc[idxs,:]
          }
        else:
          nodes_dataset[node_id]["x"] = pd.concat(
            [nodes_dataset[node_id]["x"], node_type_data.loc[idxs,:]]
          )
          nodes_dataset[node_id]["y"] = pd.concat(
            [nodes_dataset[node_id]["y"], node_type_targets.loc[idxs,:]]
          )
    # remainder data go into test
    for overload_status in [0, 1]:
      if remainder[overload_status] > 0:
        if node_type not in surely_test_set:
          surely_test_set[node_type] = {
            "x": pd.DataFrame(), "y": pd.DataFrame()
          }
        idxs = node_type_targets[
          node_type_targets["overloaded_node"] == overload_status
        ].iloc[-remainder[overload_status]:].index
        surely_test_set[node_type]["x"] = pd.concat(
          [surely_test_set[node_type]["x"], node_type_data.loc[idxs,:]]
        )
        surely_test_set[node_type]["y"] = pd.concat(
          [surely_test_set[node_type]["y"], node_type_targets.loc[idxs,:]]
        )
  # check coherence
  for node_id, node_data in nodes_dataset.items():
    if (node_data["x"].index != node_data["y"].index).any():
      raise RuntimeError(f"Incoherent dataset for node {node_id}")
  return nodes_dataset, surely_test_set

## TEST RESULTS

In [ ]:
  path_to_csvs = "../data/raw/source_domain/"
  path_to_networks = "/Users/kingsley/Documents/TESI/GITHUB_TESI/data/networks"
  base_output_folder = "../experiments"
  node_types = ["LIGHT", "MID", "HEAVY"]
  n = 20
  k = 19
  seed = 4850
  simulations = range(10)
    # for simulation in simulations:
    # print(f"{20*'-'} {simulation} {20*'-'}")
    # prepare_data(
    #   node_types, 
    #   path_to_csvs, 
    #   path_to_networks, 
    #   base_output_folder, 
    #   n, 
    #   k, 
    #   seed, 
    #   simulation, 
    #   0.1, 
    #   0.1
    # )

In [27]:
  towers_file = os.path.join(
    path_to_networks, f"porto_{n}n_{k}k/towers.csv"
  )

In [28]:
print(towers_file)

../data/networks/porto_20n_19k/towers.csv


In [ ]:
for simulation in simulations:
    print(f"{20*'-'} {simulation} {20*'-'}")
    load_network(
        path_to_networks, 
        n, 
        k, 
        seed, 
        simulation, 
        rng,
        node_types = node_types
        # functions = [col for col in multi_target_x if col.startswith('rate')],
        # n_functions_per_node = 3
    )

In [29]:
for simulation in simulations:
    print(f"{simulation}")


0
1
2
3
4
5
6
7
8
9


In [35]:
rng = np.random.default_rng(4850)

In [ ]:
/Users/kingsley/Documents/TESI/GITHUB_TESI/data/networks/porto_20n_19k/0
/Users/kingsley/Documents/TESI/GITHUB_TESI/data/networks/porto_20n_19k/towers.csv'
/Users/kingsley/Documents/TESI/GITHUB_TESI/data/networks/porto_20n_19k/seed4850/5/towers.csv'
/Users/kingsley/Documents/TESI/GITHUB_TESI/data/networks/porto_20n_19k/seed4850/5/towers.csv'
'/Users/kingsley/Documents/TESI/GITHUB_TESI/data/networks/porto_20n_19k/seed4850/5/towers.csv'

In [46]:
path_to_networks

'/Users/kingsley/Documents/TESI/GITHUB_TESI/data/networks'

In [51]:
  towers_file = os.path.join(
    path_to_networks, f"porto_{n}n_{k}k/towers.csv"
  )

In [52]:
towers_file

'/Users/kingsley/Documents/TESI/GITHUB_TESI/data/networks/porto_20n_19k/towers.csv'

In [61]:
network = load_network(
    path_to_networks, 
    n, 
    k, 
    #seed, 
    5, 
    rng,
    node_types = node_types
    # functions = [col for col in multi_target_x if col.startswith('rate')],
    # n_functions_per_node = 3
)

In [62]:
network

,lon,lat,node_type
0,-8.616455,41.144606,0
1,-8.630447,41.161652,2
2,-8.604584,41.154099,1
3,-8.631821,41.159592,1
4,-8.611336,41.157532,0
5,-8.588316,41.151401,1
6,-8.602597,41.162936,1
7,-8.624954,41.144485,1
8,-8.610829,41.142720,0
9,-8.596115,41.145859,1


the load_network works but the network i have created doesn't have a seed ???

In [ ]:
    #  nodes_dataframe, test_set = build_nodes_dataframe(
    #   multi_target_x, multi_target_y, network
    # )

In [71]:
towers

NameError: name 'towers' is not defined

In [ ]:
network = load_network(
    path_to_networks, 
    n, 
    k, 
    #seed, 
    5, 
    rng,
    node_types = node_types
    # functions = [col for col in multi_target_x if col.startswith('rate')],
    # n_functions_per_node = 3
)
nodes_dataset, test_set = build_nodes_dataframe(data_x, data_y, network)

In [72]:
nodes_dataset, test_set = build_nodes_dataframe(data_x, data_y, network)

In [73]:
nodes_dataset

{0: {'x':        node_type  rate_function_curl  rate_function_eat_memory  \
  79349          0                 0.0                       0.0   
  79350          0                 0.0                       0.0   
  79351          0                 0.0                       0.0   
  79352          0                 0.0                       0.0   
  79353          0                 0.0                       0.0   
  ...          ...                 ...                       ...   
  80617          0               110.0                      40.0   
  80618          0               110.0                      40.0   
  80631          0               120.0                      40.0   
  80632          0               120.0                      40.0   
  80633          0               120.0                      40.0   
  
         rate_function_env  rate_function_figlet  rate_function_nmap  \
  79349               10.0                 130.0                20.0   
  79350               10.0   

In [84]:
print(nodes_dataset.keys())

dict_keys([0, 4, 8, 10, 11, 12, 13, 14, 16, 2, 3, 5, 6, 7, 9, 17, 19, 1, 15, 18])


In [83]:
nodes_dataset

{0: {'x':        node_type  rate_function_curl  rate_function_eat_memory  \
  79349          0                 0.0                       0.0   
  79350          0                 0.0                       0.0   
  79351          0                 0.0                       0.0   
  79352          0                 0.0                       0.0   
  79353          0                 0.0                       0.0   
  ...          ...                 ...                       ...   
  80617          0               110.0                      40.0   
  80618          0               110.0                      40.0   
  80631          0               120.0                      40.0   
  80632          0               120.0                      40.0   
  80633          0               120.0                      40.0   
  
         rate_function_env  rate_function_figlet  rate_function_nmap  \
  79349               10.0                 130.0                20.0   
  79350               10.0   

In [85]:
nodes_dataset[0]

{'x':        node_type  rate_function_curl  rate_function_eat_memory  \
 79349          0                 0.0                       0.0   
 79350          0                 0.0                       0.0   
 79351          0                 0.0                       0.0   
 79352          0                 0.0                       0.0   
 79353          0                 0.0                       0.0   
 ...          ...                 ...                       ...   
 80617          0               110.0                      40.0   
 80618          0               110.0                      40.0   
 80631          0               120.0                      40.0   
 80632          0               120.0                      40.0   
 80633          0               120.0                      40.0   
 
        rate_function_env  rate_function_figlet  rate_function_nmap  \
 79349               10.0                 130.0                20.0   
 79350               10.0                 130.0

In [88]:
nodes_dataset[4]["y"]

,cpu_usage_function_shasum,ram_usage_function_shasum,medium_latency_function_shasum,cpu_usage_function_curl,ram_usage_function_curl,medium_latency_function_curl,cpu_usage_function_eat_memory,ram_usage_function_eat_memory,medium_latency_function_eat_memory,cpu_usage_node,...,cpu_usage_function_nmap,ram_usage_function_nmap,medium_latency_function_nmap,cpu_usage_function_env,ram_usage_function_env,medium_latency_function_env,cpu_usage_function_figlet,ram_usage_function_figlet,medium_latency_function_figlet,overloaded_node
81708,5.578333,1.118515e+07,11380890.0,91.912000,5.335732e+07,4.656731e+08,44.589000,5.500553e+07,1.183981e+09,282.217000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
81709,5.218000,1.120143e+07,9807629.0,88.173000,1.967218e+06,4.379291e+08,43.236000,5.195281e+07,1.186219e+09,288.568000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
81710,5.576000,1.119729e+07,11971438.0,91.516000,1.825451e+06,5.882080e+08,41.998000,5.131002e+07,1.202327e+09,284.304000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
81711,6.056000,1.123064e+07,18716905.0,73.437000,1.507912e+08,5.668301e+09,96.724000,8.277505e+07,1.591977e+09,308.972333,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
81712,5.964000,1.120026e+07,18169668.0,73.824000,1.497163e+08,5.832616e+09,93.470000,8.031764e+07,1.572263e+09,318.100000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
81517,5.863000,1.550336e+06,17680440.0,19.493000,1.533952e+06,4.098580e+08,248.231000,2.718448e+08,5.535287e+09,360.654000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
81518,5.835000,1.675264e+06,16595165.0,19.739000,1.633166e+06,4.088237e+08,250.623000,2.577382e+08,5.881161e+09,369.414000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
81534,5.644000,1.107223e+07,15658009.0,25.279000,1.894719e+06,4.149690e+08,198.849000,1.941362e+08,3.895496e+09,317.666667,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
81535,5.533000,1.109350e+07,16800969.0,25.628000,1.847296e+06,4.150614e+08,193.955000,2.018137e+08,4.144030e+09,322.650000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0


In [75]:
print(test_set.keys())

dict_keys([0, 1, 2])


In [82]:
test_set[0]['y']

,cpu_usage_function_shasum,ram_usage_function_shasum,medium_latency_function_shasum,cpu_usage_function_curl,ram_usage_function_curl,medium_latency_function_curl,cpu_usage_function_eat_memory,ram_usage_function_eat_memory,medium_latency_function_eat_memory,cpu_usage_node,...,cpu_usage_function_nmap,ram_usage_function_nmap,medium_latency_function_nmap,cpu_usage_function_env,ram_usage_function_env,medium_latency_function_env,cpu_usage_function_figlet,ram_usage_function_figlet,medium_latency_function_figlet,overloaded_node
98783,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,441.314,...,316.240667,3.123070e+08,1.753096e+10,2.871,1567630.222,14855900.0,9.786,7250379.02,20416309.0,0.0


In [ ]:
    # for simulation in simulations:
    # print(f"{20*'-'} {simulation} {20*'-'}")
    # prepare_data(
    #   node_types, 
    #   path_to_csvs, 
    #   path_to_networks, 
    #   base_output_folder, 
    #   n, 
    #   k, 
    #   seed, 
    #   simulation, 
    #   0.1, 
    #   0.1
    # )

# test extra

In [ ]:
    # "df_source": df_source,
    # "df_target": df_target,
    # "features": features,
    # "numerical_features": numerical_features,
    # "tasks": tasks,
    # "tasks_unified": tasks_unified,
    # "feature_dataset_source": feature_dataset_source,
    # "target_dataset_source": target_dataset_source,
    # "feature_dataset_target": feature_dataset_target,
    # "target_dataset_target": target_dataset_target,
    

In [63]:
result["df_source"]

,node_type,rate_function_curl,rate_function_eat_memory,rate_function_env,rate_function_figlet,rate_function_nmap,rate_function_shasum,cpu_usage_function_shasum,ram_usage_function_shasum,medium_latency_function_shasum,...,ram_usage_function_nmap,medium_latency_function_nmap,cpu_usage_function_env,ram_usage_function_env,medium_latency_function_env,cpu_usage_function_figlet,ram_usage_function_figlet,medium_latency_function_figlet,overloaded_status,overloaded_ratio
0,2,60.0,0.0,0.0,0.0,0.0,100.0,7.691,4042069.333,37454241.5,...,0.000000e+00,0.000000e+00,0.000000,0.000000e+00,0.000000e+00,0.000000,0.000,0.000000e+00,0,0.0
1,2,60.0,0.0,0.0,0.0,0.0,100.0,7.751,9199274.667,16880204.0,...,0.000000e+00,0.000000e+00,0.000000,0.000000e+00,0.000000e+00,0.000000,0.000,0.000000e+00,0,0.0
2,2,60.0,0.0,0.0,0.0,0.0,100.0,8.378,9214634.667,35575151.0,...,0.000000e+00,0.000000e+00,0.000000,0.000000e+00,0.000000e+00,0.000000,0.000,0.000000e+00,0,0.0
3,2,60.0,10.0,0.0,0.0,0.0,100.0,8.261,1366357.333,76360433.0,...,0.000000e+00,0.000000e+00,0.000000,0.000000e+00,0.000000e+00,0.000000,0.000,0.000000e+00,1,1.0
4,2,60.0,10.0,0.0,0.0,0.0,100.0,7.820,1323008.000,87032021.0,...,0.000000e+00,0.000000e+00,0.000000,0.000000e+00,0.000000e+00,0.000000,0.000,0.000000e+00,1,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
98779,0,0.0,0.0,20.0,70.0,70.0,0.0,0.000,0.000,0.0,...,2.022266e+08,1.655140e+10,2.024333,1.616555e+06,1.278133e+07,10.196000,5607879.111,1.611891e+07,0,0.0
98780,0,0.0,0.0,20.0,70.0,70.0,0.0,0.000,0.000,0.0,...,3.036606e+08,1.697898e+10,2.011000,1.553067e+06,1.207876e+07,10.012000,1713152.000,1.596136e+07,0,0.0
98781,0,0.0,0.0,30.0,70.0,70.0,0.0,0.000,0.000,0.0,...,2.909239e+08,1.728170e+10,2.889333,1.520640e+06,1.444850e+07,9.990167,7250379.020,1.522533e+07,0,0.0
98782,0,0.0,0.0,30.0,70.0,70.0,0.0,0.000,0.000,0.0,...,2.953576e+08,1.793603e+10,2.822000,1.727829e+06,1.444850e+07,9.619000,7250379.020,2.362620e+07,0,0.0


In [64]:
result["feature_dataset_source"]

{'Multi_Task':        node_type  rate_function_curl  rate_function_eat_memory  \
 0              2                60.0                       0.0   
 1              2                60.0                       0.0   
 2              2                60.0                       0.0   
 3              2                60.0                      10.0   
 4              2                60.0                      10.0   
 ...          ...                 ...                       ...   
 98779          0                 0.0                       0.0   
 98780          0                 0.0                       0.0   
 98781          0                 0.0                       0.0   
 98782          0                 0.0                       0.0   
 98783          0                 0.0                       0.0   
 
        rate_function_env  rate_function_figlet  rate_function_nmap  \
 0                    0.0                   0.0                 0.0   
 1                    0.0             

In [65]:
result["feature_dataset_source"]["Multi_Task"]

,node_type,rate_function_curl,rate_function_eat_memory,rate_function_env,rate_function_figlet,rate_function_nmap,rate_function_shasum
0,2,60.0,0.0,0.0,0.0,0.0,100.0
1,2,60.0,0.0,0.0,0.0,0.0,100.0
2,2,60.0,0.0,0.0,0.0,0.0,100.0
3,2,60.0,10.0,0.0,0.0,0.0,100.0
4,2,60.0,10.0,0.0,0.0,0.0,100.0
...,...,...,...,...,...,...,...
98779,0,0.0,0.0,20.0,70.0,70.0,0.0
98780,0,0.0,0.0,20.0,70.0,70.0,0.0
98781,0,0.0,0.0,30.0,70.0,70.0,0.0
98782,0,0.0,0.0,30.0,70.0,70.0,0.0


In [66]:
print(result["feature_dataset_source"].keys())

dict_keys(['Multi_Task'])


In [67]:
data_x = result["feature_dataset_source"]["Multi_Task"]
data_y = result["target_dataset_source"]["Multi_Task"]

In [68]:
(data_x.index == data_y.index).all()

np.True_

In [80]:
result["target_dataset_source"]["Multi_Task"]

,cpu_usage_function_shasum,ram_usage_function_shasum,medium_latency_function_shasum,cpu_usage_function_curl,ram_usage_function_curl,medium_latency_function_curl,cpu_usage_function_eat_memory,ram_usage_function_eat_memory,medium_latency_function_eat_memory,cpu_usage_node,...,cpu_usage_function_nmap,ram_usage_function_nmap,medium_latency_function_nmap,cpu_usage_function_env,ram_usage_function_env,medium_latency_function_env,cpu_usage_function_figlet,ram_usage_function_figlet,medium_latency_function_figlet,overloaded_node
0,7.691,4042069.333,37454241.5,28.3385,1.003324e+08,9.586719e+08,0.000000,0.000000e+00,0.000000e+00,124.895875,...,0.000000,0.000000e+00,0.000000e+00,0.000000,0.000000e+00,0.000000e+00,0.000000,0.000,0.000000e+00,0.0
1,7.751,9199274.667,16880204.0,28.1200,1.003324e+08,6.293711e+08,0.000000,0.000000e+00,0.000000e+00,127.617000,...,0.000000,0.000000e+00,0.000000e+00,0.000000,0.000000e+00,0.000000e+00,0.000000,0.000,0.000000e+00,0.0
2,8.378,9214634.667,35575151.0,28.8260,1.003324e+08,1.048772e+09,0.000000,0.000000e+00,0.000000e+00,121.317000,...,0.000000,0.000000e+00,0.000000e+00,0.000000,0.000000e+00,0.000000e+00,0.000000,0.000,0.000000e+00,0.0
3,8.261,1366357.333,76360433.0,21.1070,2.114856e+06,1.016027e+10,21.921667,8.959594e+07,1.436266e+10,145.871333,...,0.000000,0.000000e+00,0.000000e+00,0.000000,0.000000e+00,0.000000e+00,0.000000,0.000,0.000000e+00,1.0
4,7.820,1323008.000,87032021.0,20.5060,1.466204e+06,1.099271e+10,21.635000,9.420470e+07,1.472931e+10,142.147000,...,0.000000,0.000000e+00,0.000000e+00,0.000000,0.000000e+00,0.000000e+00,0.000000,0.000,0.000000e+00,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
98779,0.000,0.000,0.0,0.0000,0.000000e+00,0.000000e+00,0.000000,0.000000e+00,0.000000e+00,403.963000,...,301.279000,2.022266e+08,1.655140e+10,2.024333,1.616555e+06,1.278133e+07,10.196000,5607879.111,1.611891e+07,0.0
98780,0.000,0.000,0.0,0.0000,0.000000e+00,0.000000e+00,0.000000,0.000000e+00,0.000000e+00,398.934000,...,300.652667,3.036606e+08,1.697898e+10,2.011000,1.553067e+06,1.207876e+07,10.012000,1713152.000,1.596136e+07,0.0
98781,0.000,0.000,0.0,0.0000,0.000000e+00,0.000000e+00,0.000000,0.000000e+00,0.000000e+00,439.752000,...,307.569000,2.909239e+08,1.728170e+10,2.889333,1.520640e+06,1.444850e+07,9.990167,7250379.020,1.522533e+07,0.0
98782,0.000,0.000,0.0,0.0000,0.000000e+00,0.000000e+00,0.000000,0.000000e+00,0.000000e+00,430.189833,...,316.240667,2.953576e+08,1.793603e+10,2.822000,1.727829e+06,1.444850e+07,9.619000,7250379.020,2.362620e+07,0.0
